# 01 – Data inventory: what is in the Park et al. 2020 thymus files?

**Question.** Before any subsetting or CausalCCC run, which files should we use, and what do they actually contain?

Dataset: Park *et al.*, *A cell atlas of human thymic development defines T cell repertoire formation*, Science 2020
([PMC7611066](https://pmc.ncbi.nlm.nih.gov/articles/PMC7611066/)); data from [Zenodo 5500511](https://zenodo.org/records/5500511)
and the journal supplement (Tables S1–S5).

We check, in order:
1. the expression matrices (normalised vs raw counts) and whether their cells match;
2. the three metadata sources (h5ad `.obs`, the matching `.csv`, `sample_metadata_fix.xlsx`);
3. the study design: donors, ages, 10x chemistry, cell sorting;
4. which candidate early-niche cell populations exist, and which donors they come from;
5. Table S5 (CellPhoneDB) and how its cell-type labels relate to the atlas labels;
6. the early-niche ligand–receptor claims we will later revisit with CausalCCC.

All h5ad files are opened in *backed* mode: only metadata are read into memory.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

ROOT = Path("../..")
ZEN = ROOT / "data/thymus"
SUPP = ROOT / "data/thymus"

annot = ad.read_h5ad(ZEN / "thymus_annotated_matrix_files/HTA08.v01.A05.Science_human_fig1.h5ad", backed="r")
raw = ad.read_h5ad(ZEN / "HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")
print(annot)
print(raw)

/tmp/ipykernel_175842/2569437112.py:13: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  annot = ad.read_h5ad(ZEN / "thymus_annotated_matrix_files/HTA08.v01.A05.Science_human_fig1.h5ad", backed="r")


AnnData object with n_obs × n_vars = 255901 × 33694 backed at '../../data/thymus/zenodo_5500511/thymus_annotated_matrix_files/HTA08.v01.A05.Science_human_fig1.h5ad'
    obs: 'Anno_level_1', 'Anno_level_2', 'Anno_level_3', 'Anno_level_4', 'Anno_level_5', 'Anno_level_fig1', 'Sample', 'donor', 'organ', 'sort', 'method', 'file', 'Anno_stage', 'Age', 'Gender', 'Source'
    obsm: 'X_umap'
    layers: None (.X)
AnnData object with n_obs × n_vars = 272554 × 33694 backed at '../../data/thymus/zenodo_5500511/HTA07.A01.v02.entire_data_raw_count.h5ad'
    obs: 'Sample', 'n_counts', 'n_genes', 'donor', 'organ', 'sort', 'method', 'file', 'mito', 'doublet_scores', 'predicted_doublets', 'name'
    var: 'GeneName', 'GeneID'
    layers: None (.X)


/tmp/ipykernel_175842/2569437112.py:14: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  raw = ad.read_h5ad(ZEN / "HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")


## 1. Expression matrices

`fig1.h5ad` is the annotated object behind the paper's Figure 1 (all thymus cells). `entire_data_raw_count.h5ad` is the full
dataset before annotation. We check what kind of values each matrix holds, using the first 200 cells.

In [2]:
def describe_values(adata, n=200):
    x = adata.X[:n].toarray()
    nz = x[x != 0]
    return pd.Series({
        "max value": x.max(),
        "fraction of non-zero values that are non-integer": np.mean(nz != np.round(nz)),
        "mean total per cell": x.sum(1).mean(),
    })

pd.DataFrame({"annotated (fig1)": describe_values(annot), "raw counts": describe_values(raw)})

,annotated (fig1),raw counts
max value,9.689452,790.000000
fraction of non-zero values that are non-integer,1.000000,0.000000
mean total per cell,6996.086426,10648.509766


In [3]:
# If the annotated matrix is log1p(normalised counts), undoing the log gives the same total for every cell.
np.expm1(annot.X[:5].toarray()).sum(axis=1)

array([100000.  ,  99999.99, 100000.  , 100000.  ,  99999.99],
      dtype=float32)

**Reading.** The annotated matrix is `log1p` of counts normalised to 100,000 per cell; it holds no raw counts and has no
`.raw` slot. The raw-count file holds integers. CausalCCC/MIIC work on discretised expression, and we want to control the
normalisation ourselves, so we keep **raw counts** for the expression values and take **annotations** from the fig1 object.
That works only if the cells match:

In [4]:
shared = annot.obs_names.intersection(raw.obs_names)
print("cells in annotated object:", annot.n_obs)
print("of these, found in raw-count object:", len(shared))
print("same genes in the same order:", bool((annot.var_names == raw.var_names).all()))

agree = {c: (annot.obs.loc[shared, c].astype(str) == raw.obs.loc[shared, c].astype(str)).mean()
         for c in ["Sample", "donor", "sort", "method"]}
print("metadata agreement on shared cells:", agree)

cells in annotated object: 255901
of these, found in raw-count object: 255901
same genes in the same order: True


metadata agreement on shared cells: {'Sample': np.float64(1.0), 'donor': np.float64(1.0), 'sort': np.float64(1.0), 'method': np.float64(1.0)}


In [5]:
# Cells present only in the raw file: by sort, and how many were flagged as doublets
raw_only = raw.obs.loc[~raw.obs_names.isin(annot.obs_names)]
print(len(raw_only), "raw-only cells;", int(raw_only.predicted_doublets.sum()), "predicted doublets")
raw_only["sort"].value_counts()

16653 raw-only cells; 3592 predicted doublets


sort
CD137      3317
TOT        3009
MAIT       2640
45NM       2334
EPCAM      2122
CD3N       1035
45P        1014
45N         671
CD137NA     334
CD137PA     117
CD3P         60
Name: count, dtype: int64

Every annotated cell is in the raw file, with identical sample/donor/sort/chemistry labels. The ~16.7k extra raw cells
are QC-removed cells (doublets etc.) and sorts not used in Figure 1 (CD137, MAIT). They have no annotation, so they are
not used.

## 2. Metadata sources

In [6]:
csv = pd.read_csv(ZEN / "thymus_annotated_matrix_files/HTA08.v01.A05.Science_human_fig1.csv", index_col=0)
print("CSV has the same cells, same order:", csv.index.equals(annot.obs_names))
print("CSV has the same columns:", list(csv.columns) == list(annot.obs.columns))
print("all values identical:", all((csv[c].astype(str).values == annot.obs[c].astype(str).values).all() for c in csv.columns))

CSV has the same cells, same order: True
CSV has the same columns: True


all values identical: True


In [7]:
xlsx = pd.read_excel(ZEN / "sample_metadata_fix.xlsx", header=1)   # Table S1; row 0 is a title
print("columns:", list(xlsx.columns))
print("samples in xlsx:", xlsx.Sample.nunique(), "| in h5ad:", annot.obs.Sample.nunique(),
      "| same set:", set(xlsx.Sample) == set(annot.obs.Sample))

per_sample = annot.obs.astype(str).groupby("Sample")[["method", "sort"]].first()
cmp = per_sample.join(xlsx.set_index("Sample")[["Method", "Sort"]])
print("sort agrees with h5ad:  ", (cmp["sort"] == cmp["Sort"]).mean())
print("method agrees with h5ad:", round((cmp["method"] == cmp["Method"]).mean(), 2))

columns: ['FileName', 'NumCells', 'MeanCountsPerCell', 'MeanGenesPerCell', 'MedianMitoPercentage', 'Gender', 'Sample', 'Organ', 'DonorID', 'Sort', 'Method', 'VDJ_file', 'Enzyme']
samples in xlsx: 74 | in h5ad: 74 | same set: True
sort agrees with h5ad:   1.0
method agrees with h5ad: 0.49


In [8]:
# Which one is right? Sample names ending in "_5GEX" were sequenced with 10x 5' chemistry.
name_says_5p = per_sample.index.str.contains("5GEX")
pd.crosstab(name_says_5p, [per_sample["method"], cmp["Method"]],
            rownames=["name contains 5GEX"], colnames=["h5ad method", "xlsx Method"])

h5ad method        3GEX      5GEX     
xlsx Method        3GEX 5GEX 3GEX 5GEX
name contains 5GEX                    
False                17   19    0    0
True                  0    0   19   19

**Reading.**
- The CSV is an exact copy of the h5ad `.obs`; it adds nothing.
- The xlsx (Table S1) has **no age column**, and its `Method` column agrees with the h5ad only about half of the time,
  which is chance level. The h5ad `method` matches the sample names perfectly. So the xlsx `Method` column is unreliable;
  it is only useful for per-sample QC figures and the dissociation enzyme.
- **Decision:** use the h5ad `.obs` (for example `Age`, `method`, `sort`, `Anno_level_5`) as the single metadata source.

## 3. Study design: donors, age, chemistry, sorting

In [9]:
def age_in_weeks(a):
    # 'w' = post-conception weeks; 'm' = postnatal months; 'y' = postnatal years (birth ~ 40 weeks)
    v = float(a[:-1])
    return v if a.endswith("w") else 40 + v * 4.35 if a.endswith("m") else 40 + v * 52

obs = annot.obs.astype(str)   # plain text columns (categoricals cannot be compared across tables)
design = obs.groupby("donor", observed=True).agg(
    Age=("Age", "first"), sex=("Gender", "first"), source=("Source", "first"),
    chemistry=("method", lambda s: ", ".join(sorted(s.unique()))),
    sorts=("sort", lambda s: ", ".join(sorted(s.unique()))),
    n_cells=("Age", "size"),
)
design = design.iloc[design.Age.map(age_in_weeks).argsort()]
design

,Age,sex,source,chemistry,sorts,n_cells
donor,,,,,,
C40,7w,Female,Cambridge,3GEX,TOT,14021
C41,8w,Male,Cambridge,3GEX,TOT,11409
F22,9w,Female,HDBR,3GEX,TOT,3137
C34,9w,Female,Cambridge,5GEX,TOT,390
F74,10w,Male,HDBR,5GEX,TOT,7410
F23,11w,Male,HDBR,3GEX,"45N, 45P",5774
F64,11w,Male,HDBR,5GEX,TOT,7739
F67,12w,Female,HDBR,5GEX,TOT,13579
F45,12w,Female,HDBR,"3GEX, 5GEX","45N, 45P",20010


In [10]:
# Anno_stage = '<cell type>_<stage>'. Does the stage suffix carry anything beyond age?
stage = obs.Anno_stage.str.rsplit("_", n=1).str[-1]
ages = sorted(obs.Age.unique(), key=age_in_weeks)
pd.crosstab(stage, obs.Age)[ages]

Age,7w,8w,9w,10w,11w,12w,13w,14w,16w,17w,3m,6m,10m,15m,30m,13y,24y,35y
Anno_stage,,,,,,,,,,,,,,,,,,
early,14021,11409,3527,7410,13513,0,0,0,0,0,0,0,0,0,0,0,0,0
late,0,0,0,0,0,0,0,0,0,0,0,0,0,10811,19423,12423,22671,22804
middle,0,0,0,0,0,33589,9034,12734,18082,15078,12295,12288,4789,0,0,0,0,0


**Reading.**
- 15 fetal donors (7–17 PCW) and 8 postnatal donors (3 months – 35 years); each donor has a single age.
- The `Anno_stage` suffix is only an age bin (early = 7–11 PCW; middle = 12–17 PCW plus 3–10 months; late ≥ 15 months).
  It is **not** a separate cell-state annotation.
- Sorting differs between donors: some samples are total thymus (`TOT`), others are split into CD45⁺/CD45⁻ (`45P`/`45N`)
  or EPCAM⁺ fractions. Sorting strongly changes which cell types are captured.

## 4. Candidate early-niche populations, per fetal donor sample

The candidate senders and receivers are stromal populations (TECs, fibroblasts, VSMC, endothelium) and early lymphoid
populations (ETP, DN, ILC3, NK, γδT). Each row is one donor × chemistry × sort combination.

In [11]:
pops = ["cTEC", "mcTEC", "mTEC(I)", "mTEC(II)", "TEC(neuro)", "Fb_1", "Fb_2", "Fb_cycling", "VSMC", "Endo",
        "ETP", "DN(early)", "DN(P)", "DN(Q)", "ILC3", "NK", "γδT", "Mac", "DC1"]
fetal = obs[obs.Age.str.endswith("w")]
row = fetal.Age.astype(str) + " | " + fetal.donor.astype(str) + " | " + fetal.method.astype(str) + " | " + fetal.sort.astype(str)
tab = pd.crosstab(row, fetal.Anno_level_5).reindex(columns=pops, fill_value=0)
tab = tab.iloc[tab.index.str.split(" | ").str[0].map(age_in_weeks).argsort(kind="stable")]
tab.index.name = "age | donor | chemistry | sort"
tab

Anno_level_5,cTEC,mcTEC,mTEC(I),mTEC(II),TEC(neuro),Fb_1,Fb_2,Fb_cycling,VSMC,Endo,ETP,DN(early),DN(P),DN(Q),ILC3,NK,γδT,Mac,DC1
age | donor | chemistry | sort,,,,,,,,,,,,,,,,,,,
7w | C40 | 3GEX | TOT,5402,0,0,310,45,3,0,4,107,73,99,86,2799,1498,214,659,843,280,229
8w | C41 | 3GEX | TOT,3528,0,0,11,31,2,0,7,74,47,11,23,1108,1430,11,52,435,102,81
9w | C34 | 5GEX | TOT,60,0,1,0,2,37,13,17,8,11,0,9,28,8,2,1,22,5,6
9w | F22 | 3GEX | TOT,8,7,0,0,0,112,56,28,13,15,2,2,213,345,14,59,63,20,14
10w | F74 | 5GEX | TOT,21,20,1,1,1,126,136,48,12,30,4,19,329,632,29,61,100,68,28
11w | F23 | 3GEX | 45N,4,1,0,0,0,331,81,57,25,8,8,30,617,773,1,9,5,0,5
11w | F23 | 3GEX | 45P,0,0,0,0,0,23,31,25,32,9,3,1,8,5,17,83,41,57,13
11w | F64 | 5GEX | TOT,7,4,2,0,2,188,55,39,9,40,0,31,598,638,6,71,9,53,24
12w | F45 | 3GEX | 45N,0,1,2,0,1,157,33,6,14,4,20,20,1992,4921,3,17,2,0,1


In [12]:
# Share of each population contributed by the two youngest donors (C40 at 7 PCW, C41 at 8 PCW), among 7-11 PCW cells
early = fetal[fetal.Age.map(age_in_weeks) <= 11]
n_early = early.Anno_level_5.value_counts().reindex(pops)
share_c40_c41 = early[early.donor.isin(["C40", "C41"])].Anno_level_5.value_counts().reindex(pops, fill_value=0) / n_early
pd.DataFrame({"cells at 7-11 PCW": n_early, "share from C40 + C41": share_c40_c41.round(2)})

,cells at 7-11 PCW,share from C40 + C41
Anno_level_5,,
cTEC,9030,0.99
mcTEC,32,0.00
mTEC(I),4,0.00
mTEC(II),322,1.00
TEC(neuro),81,0.94
Fb_1,822,0.01
Fb_2,372,0.00
Fb_cycling,225,0.05
VSMC,280,0.65


**Reading – the main confounding problem.**
- **cTECs come almost entirely from two donors (C40, 7 PCW; C41, 8 PCW).** Both are from the same source (Cambridge),
  3′ chemistry and unsorted (`TOT`). This matches the paper ("cTECs were more abundant during early development, 7 to 8 PCW"),
  but it also means that in our data *cTEC identity, donor, age, source and chemistry cannot be separated*.
- Fibroblasts, by contrast, come mainly from **other** donors (CD45⁻ sorts at 11 PCW and later). Endothelium and VSMC are
  small populations (a few hundred cells) spread across donors.
- ETP and DN(early) are small (~100–200 cells at 7–11 PCW).
- So, if MIIC compares cells pooled across donors, a gene that differs between C40/C41 and the others for technical
  reasons may look like a "cTEC vs fibroblast" signal. We need to address this before running CausalCCC, for example by
  restricting to the donors that contain all populations, or by adding donor/chemistry as variables.

## 5. Table S5 (CellPhoneDB)

In [13]:
s5 = pd.read_csv(SUPP / "aay3224_table-s5.csv", header=1, low_memory=False)
annotation_cols = list(s5.columns[:10])
pair_cols = list(s5.columns[10:])
print("annotation columns:", annotation_cols)
print("interactions:", len(s5), "| cell-type pair columns:", len(pair_cols), "=", int(np.sqrt(len(pair_cols))), "squared")

# Columns are '<sender>_<receiver>'. The first 53 columns all have the first cell type as sender.
first = pair_cols[0][: len(pair_cols[0]) // 2]
s5_types = [c[len(first) + 1:] for c in pair_cols[:53]]
assert pair_cols[0] == first + "_" + first

annotation columns: ['Unnamed: 0', 'interacting_pair', 'id_cp_interaction', 'partner_a', 'partner_b', 'ensembl_a', 'ensembl_b', 'source', 'secreted', 'is_integrin']
interactions: 710 | cell-type pair columns: 2809 = 53 squared


In [14]:
atlas_types = set(obs.Anno_level_5.unique())
greek = {"CD8aa(I)": "CD8αα(I)", "CD8aa(II)": "CD8αα(II)", "abT(entry)": "αβT(entry)", "gdT": "γδT"}
s5_norm = {greek.get(t, t) for t in s5_types}
print("in S5 only:   ", sorted(s5_norm - atlas_types))
print("in atlas only:", sorted(atlas_types - s5_norm))

in S5 only:    ['B_mature', 'Epi_PAX8', 'Fb_early', 'Muscle_progenitors', 'NCC', 'TEC_early', 'VSMC_early', 'aDC', 'mTEC']
in atlas only: ['B_memory', 'B_naive', 'Epi_GCM2', 'aDC1', 'aDC2', 'aDC3', 'mTEC(I)', 'mTEC(II)', 'mTEC(III)', 'mTEC(IV)', 'mcTEC']


**Reading.**
- Table S5 contains only CellPhoneDB **mean expression** values, no p-values or significance calls.
- Its 53 cell types are **not** the published Anno_level_5 labels. Some atlas types are merged (`mTEC`, `aDC`, `B_mature`),
  and S5 has extra types that are not in the public object (`TEC_early`, `Fb_early`, `VSMC_early`, `NCC`,
  `Muscle_progenitors`, `Epi_PAX8`). The authors' code reads CellPhoneDB output from an internal run dated 2018-11-15
  (`05_Cellphone/181115_trial_1`), so S5 was computed on an earlier annotation we cannot fully rebuild.
- So S5 can tell us *which ligand–receptor pairs* were tested, but its cell-type columns can't be mapped one-to-one
  onto our cells.

## 6. What the paper claims about the early niche

In the authors' code (`HTA08.v01.A05.Fig1_and_Pellin_HSC.ipynb`, cells 92–96), the early-niche interaction figure is a
**dot plot of ligand and receptor expression per cell type and stage**, not a CellPhoneDB result. Its gene set is
below. The main text says:
- LTB:LTBR comes from diverse immune cells and is received by most stromal cell states;
- RANKL–RANK (TNFSF11:TNFRSF11A) is confined between ILC3 and mTEC(II);
- FGF7:FGFR2 goes from fibroblasts to TECs;
- NOTCH1 is the main Notch receptor on ETPs; cTECs and endothelial cells express JAG2 and DLL4, and other TECs JAG1.

In [15]:
case1_genes = ["LTB", "LTBR", "CD40LG", "CD40", "TNFSF11", "TNFRSF11A", "FGF7", "FGFR2",
               "NOTCH1", "NOTCH2", "NOTCH3", "NOTCH4", "JAG1", "JAG2", "DLL1", "DLL3", "DLL4"]
print("all genes present in the raw matrix:", all(g in raw.var_names for g in case1_genes))

s5_genes = set(s5.interacting_pair.str.split("_").explode())
print("genes of the claim set that never appear in an S5 pair name:", [g for g in case1_genes if g not in s5_genes])

all genes present in the raw matrix: True
genes of the claim set that never appear in an S5 pair name: []


## Summary and open decisions

| Item | Decision / finding |
|---|---|
| Expression values | raw counts from `HTA07.A01.v02.entire_data_raw_count.h5ad` (fig1 matrix is log-normalised only) |
| Cell annotation and metadata | fig1 h5ad `.obs` (`Anno_level_5`, `Age`, `method`, `sort`); CSV is a duplicate; xlsx `Method` unreliable |
| Early window | 7–11 PCW (= `Anno_stage` "early"); 7 donors |
| Main risk | cTEC almost only in C40/C41 (same source, chemistry, sort) → donor confounding |
| Table S5 | means only, different label set → use for the list of candidate pairs, not as cell-level truth |

To decide before the next notebook:
1. Which populations are senders and receivers for case 1 (for example cTEC, Fb_1, Endo → ETP, DN(early), ILC3)?
2. How to handle the donor confounding: restrict to C40 + C41 (which contain cTECs and early lymphoid cells), or pool
   donors and model donor/chemistry explicitly?
3. Whether to use the authors' claim gene set (Section 6), the S5 pairs, or both, as the candidate ligand–receptor
   interactions.